In [0]:
%sql

-- DISTRIBUCIÓN DE VARIETALES
-- Se emplea la columna tags para obtener la distribución de varietales

 WITH tags_explotados AS (
    SELECT EXPLODE(SPLIT(tags, ',')) AS tag_crudo
    FROM vitivinicola_catalog.bronze.productos_vino
    WHERE tags IS NOT NULL
)
SELECT TRIM(tag_crudo) AS tag_unico, COUNT(*) as cantidad
FROM tags_explotados
GROUP BY 1
HAVING cantidad > 5
ORDER BY cantidad DESC;

-- Existe dispersión en la columna tags, con presencia de caracteres corruptos y términos operativos. 
-- Requiere normalización.

In [0]:
%sql

-- DISTRIBUCIÓN DE VARIETALES POR MERCADO, TIENDA Y CATEGORIA
-- Se crea un Diccionario de Vinos Estandar como Master de Validación contra el cual se contrastan los tags para la -- clasificación y creación del atributo Varietal (Normalización).

WITH tags_explotados AS (
    SELECT 
        pais,
        tienda,
        categoria, 
        EXPLODE(SPLIT(tags, ',')) AS tag_crudo
    FROM vitivinicola_catalog.bronze.productos_vino
    WHERE tags IS NOT NULL 
),
tags_limpios AS (
    SELECT pais, tienda, categoria, TRIM(tag_crudo) AS tag_limpio
    FROM tags_explotados
)
SELECT 
    pais,
    tienda,
    categoria, 
    CASE 
        WHEN LOWER(tag_limpio) = 'malbec' THEN 'Malbec'
        WHEN LOWER(tag_limpio) IN ('cabernetsauvignon', 'cabernet sauvignon') THEN 'Cabernet Sauvignon'
        WHEN LOWER(tag_limpio) = 'chardonnay' THEN 'Chardonnay'
        WHEN LOWER(tag_limpio) IN ('pinot noir', 'pinotnoir') THEN 'Pinot Noir'
        WHEN LOWER(tag_limpio) IN ('cabernet franc', 'cabernetfranc') THEN 'Cabernet Franc'
        WHEN LOWER(tag_limpio) IN ('torrontã©s', 'torrontes') THEN 'Torrontés'
        WHEN LOWER(tag_limpio) = 'tempranillo' THEN 'Tempranillo'
        WHEN LOWER(tag_limpio) = 'verdejo' THEN 'Verdejo'
        WHEN LOWER(tag_limpio) = 'merlot' THEN 'Merlot'
        WHEN LOWER(tag_limpio) = 'syrah' THEN 'Syrah'
        WHEN LOWER(tag_limpio) = 'cinsault' THEN 'Cinsault'
        WHEN LOWER(tag_limpio) = 'tannat' THEN 'Tannat'
        WHEN LOWER(tag_limpio) = 'macabeo' THEN 'Macabeo'
        WHEN LOWER(tag_limpio) = 'sangiovese' THEN 'Sangiovese'
        WHEN LOWER(tag_limpio) IN ('albariã±o', 'albarino') THEN 'Albariño'
        WHEN LOWER(tag_limpio) = 'blend' THEN 'Blend'
        WHEN LOWER(tag_limpio) IN ('champagne', 'espumantes', 'espumante', 'espumosos') THEN 'Champagne'
        WHEN LOWER(tag_limpio) = 'pinot gris' THEN 'Pinot Gris'
        WHEN LOWER(tag_limpio) = 'pinot grigio' THEN 'Pinot Grigio'
        WHEN LOWER(tag_limpio) = 'chenin' THEN 'Chenin Blanc'
        WHEN LOWER(tag_limpio) = 'semillon' THEN 'Semillon'
        WHEN LOWER(tag_limpio) = 'barbera' THEN 'Barbera'
        WHEN LOWER(tag_limpio) = 'nebbiolo' THEN 'Nebbiolo'
        WHEN LOWER(tag_limpio) = 'carignan' THEN 'Carignan'
        WHEN LOWER(tag_limpio) = 'garnacha' THEN 'Garnacha'
        WHEN LOWER(tag_limpio) = 'bonarda' THEN 'Bonarda'
        WHEN LOWER(tag_limpio) = 'viognier' THEN 'Viognier'
        WHEN LOWER(tag_limpio) IN ('rosados', 'rosado', 'rosã©') THEN 'Rosado'
        WHEN LOWER(tag_limpio) = 'riesling' THEN 'Riesling'
        WHEN LOWER(tag_limpio) IN ('tintos', 'tinto') THEN 'Otros Tintos'
        WHEN LOWER(tag_limpio) IN ('vermut', 'vermouth', 'vermú') THEN 'Vermut'
        ELSE 'Otros' 
    END AS varietal,
    COUNT(*) AS cantidad_productos
FROM tags_limpios
WHERE LOWER(tag_limpio) IN (
    -- Tintos comunes
    'malbec', 'cabernet sauvignon', 'cabernetsauvignon', 'merlot', 'pinot noir', 'pinotnoir',
    'syrah', 'shiraz', 'tempranillo', 'cabernet franc', 'cabernetfranc', 'bonarda', 'garnacha', 
    'sangiovese', 'cinsault', 'barbera', 'petit verdot', 'tannat', 'carignan', 'nebbiolo', 'tinto', 'tintos',
    -- Blancos comunes 
    'chardonnay', 'sauvignon blanc', 'sauvignonblanc', 'torrontes', 'torrontã©s', 'viognier', 
    'riesling', 'pinot gris', 'pinot grigio', 'albarino', 'albariã±o', 'chenin', 'semillon', 'macabeo', 'verdejo',
    -- Genéricos 
    'rosados', 'rosã©', 'rosado', 'blend', 'blancos', 'espumante', 'espumosos', 'champagne','vermut', 'vermouth', 'vermút'
)
GROUP BY 1, 2, 3, 4 
ORDER BY pais, tienda, cantidad_productos DESC;

-- Se detecta sesgo crítico en la tienda Norton: devuelve cero registros clasificados

In [0]:
%sql

-- AUDITORIA ATRIBUTO TAGS

WITH tags_separados AS (
    
    SELECT 
        tienda,
        EXPLODE(SPLIT(tags, ',')) AS tag_con_espacio
    FROM vitivinicola_catalog.bronze.productos_vino 
    WHERE tags IS NOT NULL 
),
tags_limpios AS (
    
    SELECT 
        tienda,
        TRIM(tag_con_espacio) AS tag
    FROM tags_separados
    WHERE tienda IN ('Norton', 'Susana Balbo','Exclusivas Miro', 'La Barrica', 'Ocio Wine') 
),
tags_rankeados AS (
    SELECT 
        tienda,
        tag,
        COUNT(*) AS cantidad,
        ROW_NUMBER() OVER(PARTITION BY tienda ORDER BY COUNT(*) DESC) AS ranking
    FROM tags_limpios
    GROUP BY tienda, tag
)
SELECT tienda, tag, cantidad
FROM tags_rankeados
WHERE ranking <= 15
ORDER BY tienda ASC, cantidad DESC;

-- Norton utiliza el atributo tags únicamente para lógica operativa.

In [0]:
%sql

-- AUDITORIA ATRIBUTO TITULO

SELECT 
    titulo, 
    tienda, 
    tags 
FROM vitivinicola_catalog.bronze.productos_vino 
WHERE LOWER(tienda) LIKE '%norton%' ;

--  Norton requiere una estrategia diferente, no es posible inferir tags porque que utiliza nombres de fantasía en 
--  atributo titulo. Para evitar sesgos, se requerirá un diccionario de equivalencias en la capa Gold.


In [0]:
%sql
-- DIVERSIDAD DE BODEGAS Y CATEGORÍAS EN MERCADO GLOBAL

SELECT 
    pais,
    tienda,
    COUNT(DISTINCT categoria) AS diversidad_categorias,
    COUNT(DISTINCT bodega) AS diversidad_bodegas, 
    COUNT(*) AS total_productos
FROM vitivinicola_catalog.bronze.productos_vino
GROUP BY 1, 2
ORDER BY diversidad_bodegas DESC;